# 教師なし学習

## アソシエーション分析（バスケット分析）
商品購買データを用いて，アソシエーション分析（バスケット分析）を行う

### データセットの読み込み
- 商品購買のデータを `basket_data.csv` から読み込む
  - `date`: 日付（分析には使用しない）
  - `customer_id`: 購入者
  - `item_name`: 購入商品

In [ ]:
# 商品購買データの読み込み
# (cf. https://www.kaggle.com/datasets/acostasg/random-shopping-cart)
import pandas as pd
df = pd.read_csv("basket_data.csv")
display(df)

### データ形式の変換
- 購入者 (customer_id) ごとに，購入商品 (item_name) をまとめる

In [ ]:
# customer_id 列が同じ値の行について，item_name 列の値をまとめて list にする
# (index が customer_id，値が item_name の値リストの index 付き Series になる)
dataset = df.groupby("customer_id")["item_name"].apply(list)
display(dataset)

- 購入者を行，購入商品を列とする表を作り，値の True, False で誰が何を購入したかを表現する

In [ ]:
# 商品の一覧を抽出し(.fit(dataset))，2次元配列(ndarray)に変換(.transform(dataset))
#  - 各行は購入者
#  - 各列は購入商品
#  - 表の値は，購入していれば `True`，購入していなければ `False`
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)

# 更に，インデックス(customer_id)と列名の付いたデータフレーム形式に変換
df2 = pd.DataFrame(te_ary, columns=te.columns_, index=dataset.index)
display(df2)

### 支持度の計算
- 支持度(support)は，購入者の，顧客全体に対する比率を表す
- ここでは，購入商品の組み合わせ(itemsets)ごとに支持度(support)を計算して表にする
- すべての組み合わせは莫大な数になるので，支持度(support)が0.04(4%)以上のものに限定している

In [ ]:
# 支持度の計算と表示
from mlxtend.frequent_patterns import apriori
frequent_itemsets = apriori(df2, min_support=0.04, use_colnames=True)
display(frequent_itemsets)

### アソシエーション分析の実行
- 前セルの支持度(support)の表をもとに，アソシエーション分析を行う
- 各行(アソシエーション・ルール)は，前提(antecedents)の商品を買った（以下事象$A$）人が結果(consequents)の商品を買う（以下，事象$C$）かどうかの情報を表す．
- 前提と結果は複数の商品の組み合わせもあり，組み合わせの数が膨大になるので，リフト値(lift)が1以上の組み合わせに限定している

In [ ]:
# アソシエーション分析の実行（アソシエーション・ルールの抽出）
from mlxtend.frequent_patterns import association_rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# 支持度(support)の降順に並べ替える
rules = rules.sort_values("support", ascending=False)

# 先頭の20行のみ表示
display(rules.head(20))

各列の値は以下の通り
- antecedent support : 前提の支持度
  - 前提の商品の購入者の，顧客全体に対する比率．$P(A)$．前提の商品がどのくらい売れているか
- consequent support : 結果の支持度
  - 結果の商品の購入者の，顧客全体に対する比率．$P(C)$．結果の商品がどのくらい売れているか
- <span style="color:forestgreen;border-bottom:solid 1px;">support : 支持度</span>
  - 前提と結果の両方の商品を同時に買った人の，顧客全体に対する比率．$P(A∩C)$．前提と結果の両方の商品を同時に買った顧客がどのくらいいるか
  - この値が小さい場合，めったに起こらない事象なので，あまり有効なルールではない
- <span style="color:forestgreen;border-bottom:solid 1px;">confidence : 確信度</span>
  - $\frac{P(A∩C)}{P(A)}=P(C|A)$，前提の商品の購入者のうち，結果の商品を買った顧客はどのくらいいるかの比率
  - この値が小さい場合も，あまり有効なルールではない
- <span style="color:forestgreen;border-bottom:solid 1px;">lift : リフト値</span>
  - 「確信度」の「結果の支持度」に対する比率．$\frac{P(C|A)}{P(C)}$．前提の商品を買ったという条件が加わると結果の商品を買う確率が何倍になるかということ
  - この値が ($1$以上で) 大きいほど，前提の商品の購入が結果の商品の購入に，大きな促進的影響を及ぼすと考えられる ($0$以上$1$未満なら抑制的な影響)
  - 前提と結果 ($A$と$C$) を入れ替えても値は変わらない
- leverage
  - 「支持度」から「前提の支持度」と「結果の支持度」の積を引いたもの．$P(A∩C)-P(A)P(C)$．前提事象$A$と結果事象$C$の両方が起こる確率について，実際の確率と $A,C$が独立であると仮定したときの確率との差
  - この値が$0$に近いほど，事象$A$と事象$C$は独立事象に近い
  - この値が (正で) 大きいほど，前提の商品の購入が結果の商品の購入に，大きな促進的影響を及ぼすと考えられる (負なら抑制的な影響)
  - 前提と結果 ($A$と$C$) を入れ替えても値は変わらない
- conviction
  - $\frac{1-P(C)}{1-P(C|A)}=\frac{P(\overline C)}{P(\overline C|A)}$．前提の商品を買ったという条件が加わると結果の商品を買わないという確率が何分の1になるかということ
  - リフト値と同じく，この値が ($1$以上で) 大きいほど，前提の商品の購入が結果の商品の購入に，大きな促進的影響を及ぼすと考えられる
  <!-- - 同じリフト値であっても，結果の商品が買われる確率$P(C)$が高いほど，この値は大きくなる -->

## クラスタリング (教師なし学習)
ペンギンの種類のデータは伏せて，ペンギンの体の計測データのみを用いてクラスターに分けてみる

### ペンギンのデータ (penguins) の取得と前処理<br><span style="font-size:60%;">　(参照： https://github.com/allisonhorst/palmerpenguins/blob/main/README.md)</span>

<img src="https://raw.githubusercontent.com/allisonhorst/palmerpenguins/refs/heads/main/man/figures/culmen_depth.png" style="width:600px">

-  各数値データ (bill_length_mm，bill_depth_mm，flipper_length_mm，body_mass_g) 
   について，それらを正規化した値 (平均0，標準偏差1になるように変換した値) の列
   (bill_length_Z，bill_depth_Z，flipper_length_Z，body_mass_Z) を追加しておく

In [ ]:
import requests     # Web上のデータを取得するライブラリをインポートする
import io           # 入出力用のライブラリをインポートする
import pandas as pd # pandas ライブラリを pd という名前でインポートする

# CSVファイルのURL
url_penguins = 'https://github.com/allisonhorst/palmerpenguins/raw/main/inst/extdata/penguins.csv'

# url_penguins のCSVファイルを取得して，データフレーム形式にして df_penguins に保存
df_penguins = pd.read_csv(io.BytesIO(requests.get(url_penguins).content))

# 数値データを正規化した (平均0，標準偏差1 に変換した) データ (Z得点) を追加
from sklearn import preprocessing
df_penguins['bill_length_Z'] = preprocessing.scale(df_penguins['bill_length_mm'])
df_penguins['bill_depth_Z'] = preprocessing.scale(df_penguins['bill_depth_mm'])
df_penguins['flipper_length_Z'] = preprocessing.scale(df_penguins['flipper_length_mm'])
df_penguins['body_mass_Z'] = preprocessing.scale(df_penguins['body_mass_g'])

# df_penguins に保存したデータフレームを表示
display(df_penguins)

### 使用変数の抽出
-  「くちばしの長さ」と「体重」の正規化された値 (bill_length_Z，body_mass_Z) を取り出したデータフレーム (df) を作る  
   (これをもとに，クラスタリングを行う)

In [ ]:
X_vars = ['bill_length_Z', 'body_mass_Z']
y_var = 'species'

# 必要なデータを取り出し，欠損値 (NaN) を含む行を削除
df = df_penguins[X_vars + [y_var]].dropna()

X = df[X_vars]
y = df[y_var]

# データフレームの表示
display(df)

# 散布図の表示
color_map = {'Adelie':'orangered', 'Chinstrap':'darkgreen', 'Gentoo':'darkviolet'}
_ = df.plot.scatter(
    x=X_vars[0], y=X_vars[1],
    c=df[y_var].replace(color_map),
    alpha=0.5,
)

-   <span style="background-color: orangered">　</span> Adelie（アデリーペンギン）   
-   <span style="background-color: darkgreen">　</span> Chinstrap（ヒゲペンギン）   
-   <span style="background-color: darkviolet">　</span> Gentoo（ジェンツーペンギン）   

### クラスタリングの実行
ペンギンの種類 (species) のデータ抜きで，各ペンギンを3つのクラスターに分けてみる  
クラスタリングの手法にはさまざまなものがあるが，ここでは以下の3通りの方法でクラスタリングを行う．
- <span style="color:forestgreen;border-bottom:solid 1px;">k-平均 (k-means) 法</span>　―　非階層的クラスタリング
- <span style="color:forestgreen;border-bottom:solid 1px;">ウォード (Ward) 法</span>　―　階層的クラスタリング
- (おまけ) 混合正規分布モデル (混合ガウスモデル; Gaussian mixture model; GMM) 
  - 各クラスタが正規分布になるという仮定の下で，もっとも誤差の少ないクラスタリングを推定する非階層的クラスタリング

In [ ]:
# クラスタリングの実行 (クラスターは番号で示される)

# KMeans の memory leak の回避
import os
os.environ['OMP_NUM_THREADS'] = '2'

# K-means 法
from sklearn.cluster import KMeans
kmeans = KMeans(3, random_state=0).fit(X)

# ウォード (Ward) 法
from sklearn.cluster import AgglomerativeClustering
ward = AgglomerativeClustering(3).fit(X)

# 混合正規分布モデル (GMM)
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(3, random_state=0).fit(X)

# 結果の列をデータに追加
df['kmeans'] = kmeans.predict(X)
df['ward'] = ward.labels_
df['gmm'] = gmm.predict(X)

# 表示
display(df)

-  クラスタ番号をペンギンの種類の名前に付け替える
    -   #142 は Adelie，#307 は Chinstrap，#169 は Gentoo と仮定する（場当たり的ですが……）

In [ ]:
# クラスタ番号を名前に付け替え
df.replace({
    'kmeans':{  # kmeans の列での付け換え
        df.loc[142, 'kmeans']:'Adelie', 
        df.loc[307, 'kmeans']:'Chinstrap', 
        df.loc[169, 'kmeans']:'Gentoo'},
    'ward':{    # ward の列での付け換え
        df.loc[142, 'ward']:'Adelie', 
        df.loc[307, 'ward']:'Chinstrap', 
        df.loc[169, 'ward']:'Gentoo'},
    'gmm':{     # gmm の列での付け換え
        df.loc[142, 'gmm']:'Adelie', 
        df.loc[307, 'gmm']:'Chinstrap', 
        df.loc[169, 'gmm']:'Gentoo'}}, inplace=True)

display(df)

### 結果の評価
混同行列，正解率，散布図

In [ ]:
# 混同行列と正解率
from sklearn.metrics import confusion_matrix
print(confusion_matrix(df["species"], df["kmeans"]), end='\t')
print('kmeans 正解率:', round(100*len(df[df['species']==df['kmeans']])/len(df), 1), "%")
print(confusion_matrix(df["species"], df["ward"]), end='\t')
print('  ward 正解率:', round(100*len(df[df['species']==df['ward']])/len(df), 1), "%")
print(confusion_matrix(df["species"], df["gmm"]), end='\t')
print('   gmm 正解率:', round(100*len(df[df['species']==df['gmm']])/len(df), 1), "%")

# 散布図の表示

# サブプロットを作成
import matplotlib.pyplot as plt
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 8))

# 各サブプロットに散布図を描画
color_map = {'Adelie':'orangered', 'Chinstrap':'darkgreen', 'Gentoo':'darkviolet'}
df.plot.scatter(
    x=X_vars[0], y=X_vars[1], c=df['species'].replace(color_map), alpha=0.5, title='species (correct ans.)',
    ax=axes[0, 0])
df.plot.scatter(
    x=X_vars[0], y=X_vars[1], c=df['kmeans'].replace(color_map), alpha=0.5, title='k-means',
    ax=axes[0, 1])
df.plot.scatter(
    x=X_vars[0], y=X_vars[1], c=df['ward'].replace(color_map), alpha=0.5, title='Ward',
    ax=axes[1, 0])
df.plot.scatter(
    x=X_vars[0], y=X_vars[1], c=df['gmm'].replace(color_map), alpha=0.5, title='GMM',
    ax=axes[1, 1])

plt.tight_layout()  # グラフ同士が重ならないように調整
plt.show()

## 〔参考〕正規化しなかった場合のクラスタリング
-  上と同様に「くちばしの長さ」と「体重」を用いるが，正規化する前の値 (bill_length_mm，body_mass_g) でクラスタリングを行ってみる
### 使用変数の抽出

In [ ]:
X_vars = ['bill_length_mm', 'body_mass_g']
y_var = 'species'

# 必要なデータを取り出し，欠損値 (NaN) を含む行を削除
df2 = df_penguins[X_vars + [y_var]].dropna()

X = df2[X_vars]
y = df2[y_var]

# データフレームの表示
display(df2)

#  散布図の表示（省略）
# color_map = {'Adelie':'orangered', 'Chinstrap':'darkgreen', 'Gentoo':'darkviolet'}
# _ = df.plot.scatter(
#     x=X_vars[0], y=X_vars[1],
#     c=df[y_var].replace(color_map),
#     alpha=0.5,
# )

### クラスタリングの実行

In [ ]:
# クラスタリングの実行 (クラスターは番号で示される)

# KMeans の memory leak の回避
import os
os.environ['OMP_NUM_THREADS'] = '2'

# K-means 法
from sklearn.cluster import KMeans
kmeans = KMeans(3, random_state=0).fit(X)

# ウォード (Ward) 法
from sklearn.cluster import AgglomerativeClustering
ward = AgglomerativeClustering(3).fit(X)

# 混合正規分布モデル (GMM)
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(3, random_state=0).fit(X)

# 結果の列をデータに追加
df2['kmeans'] = kmeans.predict(X)
df2['ward'] = ward.labels_
df2['gmm'] = gmm.predict(X)

# 表示
# display(df2)

# クラスタ番号を名前に付け替え
# #142 は Adelie，#307 は Chinstrap，#169 は Gentoo と仮定する（場当たり的ですが……）
df2.replace({
    'kmeans':{  # kmeans の列での付け換え
        df2.loc[142, 'kmeans']:'Adelie', 
        df2.loc[307, 'kmeans']:'Chinstrap', 
        df2.loc[169, 'kmeans']:'Gentoo'},
    'ward':{    # ward の列での付け換え
        df2.loc[142, 'ward']:'Adelie', 
        df2.loc[307, 'ward']:'Chinstrap', 
        df2.loc[169, 'ward']:'Gentoo'},
    'gmm':{     # gmm の列での付け換え
        df2.loc[142, 'gmm']:'Adelie', 
        df2.loc[307, 'gmm']:'Chinstrap', 
        df2.loc[169, 'gmm']:'Gentoo'}}, inplace=True)

display(df2)

### 結果の評価
混同行列，正解率，散布図

In [ ]:
# 混同行列と正解率
from sklearn.metrics import confusion_matrix
print(confusion_matrix(df2["species"], df2["kmeans"]), end='\t')
print('kmeans 正解率:', round(100*len(df2[df2['species']==df2['kmeans']])/len(df2), 1), "%")
print(confusion_matrix(df2["species"], df2["ward"]), end='\t')
print('  ward 正解率:', round(100*len(df2[df2['species']==df2['ward']])/len(df2), 1), "%")
print(confusion_matrix(df2["species"], df2["gmm"]), end='\t')
print('   gmm 正解率:', round(100*len(df2[df2['species']==df2['gmm']])/len(df2), 1), "%")

# 散布図の表示

# サブプロットを作成
import matplotlib.pyplot as plt
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 8))

# 各サブプロットに散布図を描画
color_map = {'Adelie':'orangered', 'Chinstrap':'darkgreen', 'Gentoo':'darkviolet'}
df2.plot.scatter(
    x=X_vars[0], y=X_vars[1], c=df2['species'].replace(color_map), alpha=0.5, title='species (correct ans.)',
    ax=axes[0, 0])
df2.plot.scatter(
    x=X_vars[0], y=X_vars[1], c=df2['kmeans'].replace(color_map), alpha=0.5, title='k-means',
    ax=axes[0, 1])
df2.plot.scatter(
    x=X_vars[0], y=X_vars[1], c=df2['ward'].replace(color_map), alpha=0.5, title='Ward',
    ax=axes[1, 0])
df2.plot.scatter(
    x=X_vars[0], y=X_vars[1], c=df2['gmm'].replace(color_map), alpha=0.5, title='GMM',
    ax=axes[1, 1])

plt.tight_layout()  # グラフ同士が重ならないように調整
plt.show()